# Short exercises: weighted metapaths on the HALD knowledge graph

Four short exercises on the notebook **`HALD_DWPC_metapath_analysis`** (about **30 minutes** in total).

| Exercise | Topic | Time |
|---|---|---|
| 1 | What does the damping exponent $w$ do? | 7 min |
| 2 | A toy example by hand | 8 min |
| 3 | An ROC curve by hand | 8 min |
| 4 | Fairer negatives | 7 min |

**How to work.** Start the **Setup** cell below first (it runs for about two minutes) and read Exercise 1 while it runs. Do the *by hand* parts on paper, then run the check cells. A cell with `...` has one line for you to complete. Random seeds are fixed, so your numbers should match the expected ones.

---
## Setup (run this first; about 2 minutes)

These cells are copied from Sections 2–7 of the main notebook (loading HALD, adjacency matrices, metapaths, DWPC functions, training pairs, features). The big debug printouts and the permutation analysis are left out. **Edit `RDF` in the first cell if your path differs.** If your kernel already holds the main notebook's variables, skip this section.

In [ ]:
import os, time, random
from collections import defaultdict
import numpy as np, pandas as pd, scipy.sparse as sp
from owlready2 import *
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------ configuration
RDF    = "./HALD_Dataset/HALD.rdf"          # path to the HALD RDF/XML file
OUT    = "results_dwpc"      # output folder (CSV results + SQLite cache)
MAXLEN = 3                   # maximum metapath length (3 = up to three metaedges)
N_PERM = 5                   # number of permuted networks
W_GRID = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]   # damping exponents to try
SEED, N_NEG = 42, 20000      # random seed; number of sampled negative pairs

os.makedirs(OUT, exist_ok=True)
random.seed(SEED); rng = np.random.default_rng(SEED)

In [ ]:
# Load the ontology; the SQLite backend caches the parsed triples so later runs start quickly.
#default_world.set_backend(filename=os.path.join(OUT, "hald.sqlite3"))
onto = get_ontology("file://" + os.path.abspath(RDF)).load()
default_world.save()

# Define a lambda function that automatically prepends the namespace prefixes we will use.
PRE = ("PREFIX onto: <http://www.dummy.info/new.owl#>\n"
       "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
       "PREFIX owl: <http://www.w3.org/2002/07/owl#>\n")
sparql = lambda q: default_world.sparql(PRE + q)

ClassCounts = list(sparql(
       """
       SELECT ?type (COUNT(DISTINCT ?node) AS ?count)
       WHERE {
              ?node a ?type
              ?type a owl:Class
       }
       GROUP BY ?type
       ORDER BY DESC(?count)
       """
))

# Extract just the classes for convenience
ClassList = [i[0] for i in ClassCounts]

# Index the nodes for later use in adjacency matrices
Nodes = dict()
for c in ClassList:
    nodelist = list(sparql(f"SELECT ?i WHERE {{ ?i a onto:{c.name} }}"))
    Nodes[c.name] = sorted(i[0].name for i in nodelist)
idx   = {c: {n: k for k, n in enumerate(v)} for c, v in Nodes.items()}
print(Nodes)

In [ ]:
Targets = list(sparql(
    """
    SELECT ?gene ?disease
    WHERE {
        ?gene a onto:Gene
        ?disease a onto:Disease
        ?gene onto:reduce ?disease
    }
    """
    ))


# Get the pairs sorted.
pos = sorted([(g.name, d.name) for g, d in Targets])
# Convert this to an index array - will be needed later during validation and testing
pos_idx = np.array([(idx["Gene"][g], idx["Disease"][d]) for g, d in pos])

print(len(pos), "positive Gene-reduce-Disease pairs")

In [ ]:
EDGES = {}     # (a, b) -> array of (row index in class a, column index in class b)

# For each pair of node types linked by an edge, get the edges
for a in ClassList:
    for b in ClassList:
        metaedges = list(sparql(
            f"""
            SELECT DISTINCT ?n1 ?n2 
            WHERE {{
                ?n1 a onto:{a.name}
                ?n2 a onto:{b.name}
                ?n1 ?p ?n2
                ?p a owl:ObjectProperty    
                FILTER( !( ?p = onto:reduce && onto:{a.name} = onto:Gene && onto:{b.name} = onto:Disease) )
            }}
            """
        ))

        # Build a list of edge pairs (nodes with an edge between them), removing self-edges and taking their index from the list we created earlier.
        edgepairs = {(idx[a.name][x.name], idx[b.name][y.name]) for x, y in metaedges if not (a.name == b.name and x.name == y.name)}
        # If there are some edges between a pair of types, put them in the master EDGES table. This is a dictionary keyed by a tuple of the two node types.
        if edgepairs:
            EDGES[(a.name, b.name)] = np.array(sorted(edgepairs))

# print some summary data
print(len(EDGES), "metaedges,", sum(len(e) for e in EDGES.values()), "distinct edges")
metaedge_table = pd.DataFrame([(a, b, len(e)) for (a, b), e in EDGES.items()],
                              columns=["src", "dst", "edges"]).sort_values("edges", ascending=False)
metaedge_table.to_csv(f"{OUT}/metaedges.csv", index=False)
metaedge_table.head(10)

In [ ]:
# For each *type* of node-node edge calculate the total number of metaedge between two nodes of that type
# Example: A0[('Gene', 'Disease')] holds
# Coords	Values
# (5, 288)	1.0
# (5, 836)	1.0
# (5, 1360)	1.0
# (5, 3465)	1.0
# (11, 123)	1.0
# (12, 1844)	1.0
# (12, 1925)	1.0
#
# which means that node 5 of type Gene has 1 connection to node 1360 of type disease etc etc


def adjacency(edges,nodes):
    X = dict()
    for k,e in edges.items():
        X[k] = sp.csr_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(Nodes[k[0]]), len(Nodes[k[1]])))
    return X


A0 = adjacency(EDGES, Nodes)

# Fetch the Gene from the table with the highest connectivity to Disease
g0 = Nodes["Gene"][int(np.argmax(np.asarray(A0[("Gene", "Disease")].sum(1))))]


# Get that node by name and count its connections, removing reduce connections
q = list(sparql(
    f'''
    SELECT (COUNT(DISTINCT ?d) AS ?n)
    WHERE {{ 
        onto:{g0} ?p ?d
        ?d rdf:type onto:Disease
        ?p a owl:ObjectProperty
        FILTER(?p != onto:reduce)
    }}
    '''
))[0][0]

# If the counts returned by the two methods don't match raise an assertion
assert q == int(A0[("Gene", "Disease")][idx["Gene"][g0]].sum()), "degree mismatch"
print(f"degree check OK (Node {g0}: Gene>Disease degree = {q})")

In [ ]:
STEPS = {}     # class -> list of (next class, direction, metaedge key)

# Set up the rules for the metapath. For each edge (a,b) the allowabvle moves are a > b or b < a
# This is effectively a linked list.
for (a, b) in EDGES:
    STEPS.setdefault(a, []).append((b, ">", (a, b)))   # forward  : uses A_ab
    STEPS.setdefault(b, []).append((a, "<", (a, b)))   # backward : uses A_ab^T


# construct the paths recursively
def enumerate_metapaths(maxlen):
    out = []
    def rec(path):
        if len(path) > 1 and path[-1][0] == "Disease": out.append(tuple(path))
        if len(path) - 1 == maxlen: return
        for nxt, d, key in STEPS.get(path[-1][0], []):
            rec(path + [(nxt, d, key)])
    rec([("Gene", "", None)])
    return out

name_of = lambda mp: mp[0][0] + "".join(d + c for c, d, _ in mp[1:])
METAPATHS = {name_of(m): m for m in enumerate_metapaths(MAXLEN)}
print(len(METAPATHS), f"metapaths (length <= {MAXLEN}; both edge directions allowed)")
list(METAPATHS)[:10]

In [ ]:
def weighted(A, w):
    # W_ab = D_out^-w  A_ab  D_in^-w  for every metaedge
    W = {}
    for k, M in A.items():
        out_d = np.asarray(M.sum(1)).ravel() # ravel flattens the array
        in_d = np.asarray(M.sum(0)).ravel()
        do = sp.diags(np.where(out_d > 0, out_d, 1.0) ** -w)
        di = sp.diags(np.where(in_d  > 0, in_d,  1.0) ** -w)
        W[k] = (do @ M @ di).tocsr()
    return W

def dwpc_matrix(mp, W):
    # full (Gene x Disease) DWPC matrix for one metapath
    mats  = [W[key] if d == ">" else W[key].T.tocsr() for _, d, key in mp[1:]]
    types = [c for c, _, _ in mp]
    M = mats[0]
    for X in mats[1:]: M = M @ X
    if len(mats) == 3:                                   # remove paths with duplicate nodes
        W1, W2, W3 = mats
        a_eq = types[0] == types[2]
        b_eq = types[1] == types[3]
        if a_eq:
            M = M - sp.diags(np.asarray(W1.multiply(W2.T).sum(1)).ravel()) @ W3
        if b_eq:
            M = M - W1 @ sp.diags(np.asarray(W2.multiply(W3.T).sum(1)).ravel())
        if a_eq and b_eq:
            M = M + W1.multiply(W2.T).multiply(W3)
    return M.tocsr()

def features(A, w, names, pairs):
    # DWPC value of each metapath in `names` for each (gene, disease) pair in `pairs`
    W = weighted(A, w); cols = []
    for n in names:
        M = dwpc_matrix(METAPATHS[n], W)
        cols.append(np.asarray(M[pairs[:, 0], pairs[:, 1]]).ravel())
    return np.column_stack(cols)

In [ ]:
def brute(A, mp, w, g, d):
    deg = {k: (np.asarray(M.sum(1)).ravel(), np.asarray(M.sum(0)).ravel()) for k, M in A.items()}
    total = 0.0
    def rec(i, node, pdp, seen):
        nonlocal total
        if i == len(mp):
            if node == d: total += pdp
            return
        c, dirn, key = mp[i];
        M = A[key] if dirn == ">" else A[key].T.tocsr()
        od, idg = deg[key] if dirn == ">" else deg[key][::-1]
        for nb in M[node].indices:
            if (c, nb) in seen: continue
            rec(i + 1, nb, pdp * od[node] ** -w * idg[nb] ** -w, seen | {(c, nb)})
    rec(1, g, 1.0, {("Gene", g)})
    return total

for chk in ["Gene>Disease>Disease", "Gene>Disease<Gene>Disease"]:     # length 2 and a length-3 path with repeats
    if chk in METAPATHS:
        Mc = dwpc_matrix(METAPATHS[chk], weighted(A0, 0.4))
        for g, d in pos_idx[:3]:
            assert abs(Mc[g, d] - brute(A0, METAPATHS[chk], 0.4, g, d)) < 1e-9
        print(f"{chk}: matrix DWPC matches brute-force enumeration")

In [ ]:
n_g, n_d = len(Nodes["Gene"]), len(Nodes["Disease"])
posset = {tuple(p) for p in pos_idx.tolist()}
neg = set()
while len(neg) < N_NEG:
    p = (int(rng.integers(n_g)), int(rng.integers(n_d)))
    if p not in posset:
        neg.add(p)

PAIRS = np.vstack([pos_idx, np.array(sorted(neg))])                       # (gene index, disease index)
Y = np.r_[np.ones(len(pos_idx)), np.zeros(N_NEG)].astype(int)
print(PAIRS.shape[0], "pairs:", Y.sum(), "positive,", (Y == 0).sum(), "negative")

class MeanArcsinh(BaseEstimator, TransformerMixin):
    # mean scaling followed by inverse hyperbolic sine (Project Rephetio)
    def fit(self, X, y=None):
        m = X.mean(0); self.m_ = np.where(m > 0, m, 1.0); return self
    def transform(self, X): return np.arcsinh(X / self.m_)

def auc_table(X, names):
    # single-feature AUROC / AP, for features connected to >= 5 pairs
    rows = []
    for j, n in enumerate(names):
        if (X[:, j] > 0).sum() >= 5 and X[Y == 1, j].any():
            rows.append((n, roc_auc_score(Y, X[:, j]), average_precision_score(Y, X[:, j]), int((X[Y == 1, j] > 0).sum())))
    return pd.DataFrame(rows, columns=["metapath", "AUC", "AP", "pos_covered"]).set_index("metapath")

In [ ]:
def make_ridge(C=0.01):
    return make_pipeline(MeanArcsinh(), StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced", max_iter=3000))

def cv_oof(Xm, mk=make_ridge):
    # 5-fold stratified CV; return out-of-fold AUROC and AP
    oof = np.zeros(len(Y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xm, Y):
        oof[te] = mk().fit(Xm[tr], Y[tr]).predict_proba(Xm[te])[:, 1]
    return roc_auc_score(Y, oof), average_precision_score(Y, oof)

short = [n for n, m in METAPATHS.items() if len(m) <= 3]      # metapaths with <= 2 metaedges
wrows = []
for w in W_GRID:
    Xs = features(A0, w, short, PAIRS); Xs = Xs[:, (Xs > 0).sum(0) >= 5]
    a, p = cv_oof(Xs)
    t = auc_table(Xs, [f"m{i}" for i in range(Xs.shape[1])])
    wrows.append((w, a, p, t.AUC.mean()))
    print(f"w={w}: CV AUC={a:.4f}  AP={p:.4f}  mean single-metapath AUC={t.AUC.mean():.4f}")

wdf = pd.DataFrame(wrows, columns=["w", "CV_AUC", "CV_AP", "mean_single_AUC"])
wdf.to_csv(f"{OUT}/damping_exponent_grid.csv", index=False)
W_BEST = float(wdf.loc[wdf.CV_AUC.idxmax(), "w"])
print("selected damping exponent w =", W_BEST)
wdf

In [ ]:
NAMES = list(METAPATHS)
X  = features(A0, W_BEST, NAMES, PAIRS)      # DWPC at the selected w
X0 = features(A0, 0.0,    NAMES, PAIRS)      # plain path count (w = 0)
keep = [j for j in range(len(NAMES)) if (X[:, j] > 0).sum() >= 5]
NAMES = [NAMES[j] for j in keep]; X, X0 = X[:, keep], X0[:, keep]
print(len(NAMES), "metapaths with >= 5 connected pairs")

single  = auc_table(X, NAMES)
single["AUC_pathcount_w0"] = auc_table(X0, NAMES).AUC

def degree_features(A, pairs):
    cols, nm = [], []
    for (a, b), M in A.items():
        if a == "Gene":    cols.append(np.asarray(M.sum(1)).ravel()[pairs[:, 0]]); nm.append(f"deg_Gene_out_{a}>{b}")
        if b == "Gene":    cols.append(np.asarray(M.sum(0)).ravel()[pairs[:, 0]]); nm.append(f"deg_Gene_in_{a}>{b}")
        if a == "Disease": cols.append(np.asarray(M.sum(1)).ravel()[pairs[:, 1]]); nm.append(f"deg_Disease_out_{a}>{b}")
        if b == "Disease": cols.append(np.asarray(M.sum(0)).ravel()[pairs[:, 1]]); nm.append(f"deg_Disease_in_{a}>{b}")
    return np.column_stack(cols).astype(float), nm

XD, DNAMES = degree_features(A0, PAIRS)
kd = XD.std(0) > 0; XD = XD[:, kd]; DNAMES = [n for n, k in zip(DNAMES, kd) if k]
print(XD.shape[1], "degree covariates")

In [ ]:
assert all(v in globals() for v in ["A0", "METAPATHS", "NAMES", "PAIRS", "Y", "X", "XD", "kd", "W_BEST", "features"]), "run the setup cells first"
print("setup OK")

---
# Exercise 1: What does the damping exponent $w$ do?  *(7 min)*

Each path is weighted by $\mathrm{PDP}(\pi)=\prod d^{-w}$ over the degrees along it, so $w=0$ is the plain path count and larger $w$ penalises hubs more.

**Q1 (by hand).** Two paths are identical except that one passes through a node of degree 4 and the other through a node of degree 400. By what factor is the second path's weight smaller than the first's when $w=0,\ 0.5,\ 1$?

**Task.** Complete the one missing line, then run the cell (about a minute). It computes the DWPC features at three values of $w$ and reports the 5-fold cross-validated AUROC and average precision (AP) of a ridge model.

In [ ]:
mk_c = lambda: make_ridge(C=1e-4)

for w in [0, 0.5, 1.0]:
    Xw = features(A0, w, NAMES, PAIRS)           # DWPC for every metapath at this w
    a, p = cv_oof(Xw, mk_c)                      # cross-validated (AUROC, AP)
    print(f"w={w}:  CV AUROC={a:.3f}   AP={p:.3f}")

**Q2.** Does $w$ change the AUROC much? Which $w$ gives the best AP, and would you trust that difference with only 132 positive pairs?

**Solution Q1.** The ratio is $(400/4)^{-w}=100^{-w}$: **1** at $w=0$, **0.1** at $w=0.5$, **0.01** at $w=1$. The hub path matters 10 to 100 times less.

**Expected output (seed 42).**

| $w$ | CV AUROC | AP |
|---|---|---|
| 0 | 0.978 | 0.746 |
| 0.5 | 0.970 | 0.769 |
| 1.0 | 0.980 | 0.794 |

**Solution Q2.** AUROC barely changes (0.970 to 0.980). AP is highest at $w=1$ (0.794 against 0.746 at $w=0$), but with 132 positives a difference of a few hundredths is within the noise (a different random seed would likely reorder them), so the honest conclusion is that on HALD the damping exponent matters little.

---
# Exercise 2: A toy example by hand  *(8 min)*

A toy graph with four genes and three diseases (use $w=0.5$):

| Gene → Disease | Disease → Disease |
|---|---|
| BRCA1 → BC | BC → AD |
| TP53 → BC | |
| TP53 → AD | |
| APOE → AD | |
| APOE → T2D | |
| **MLH1 → BC** | |
| **MLH1 → T2D** | |

(The two bold rows are new compared with the lecture example.) We look at the metapath `Gene>Disease>Disease`.

**By hand.**

1. Write the $4\times3$ adjacency matrix $A_{GD}$ (rows BRCA1, TP53, APOE, MLH1; columns BC, AD, T2D).
2. Find the out-degree of each gene and the in-degree of each disease.
3. Compute the weights $W[\text{BRCA1},\text{BC}]$ and $W[\text{MLH1},\text{BC}]$, using $W_{xy}=d^{\,\mathrm{out}}(x)^{-w}\,d^{\,\mathrm{in}}(y)^{-w}$.
4. There is exactly one path from BRCA1 to AD: BRCA1 → BC → AD. Compute its weight (the Disease → Disease edge has weight 1).

**Solution.**

1. $A_{GD}=\begin{pmatrix}1&0&0\\1&1&0\\0&1&1\\1&0&1\end{pmatrix}$
2. Gene out-degrees $(1,2,2,2)$. Disease in-degrees (BC, AD, T2D) $=(3,2,2)$.
3. $W[\text{BRCA1},\text{BC}]=1^{-1/2}\cdot3^{-1/2}=0.577$ and $W[\text{MLH1},\text{BC}]=2^{-1/2}\cdot3^{-1/2}=0.408$.
4. DWPC(BRCA1, AD) $=0.577\times1=\mathbf{0.577}$.

**Check in code.** Run this cell; it builds the matrices and uses the notebook's own `weighted` and `dwpc_matrix` functions. Does the BRCA1 → AD entry match your answer to step 4?

In [ ]:
genes_t = ["BRCA1", "TP53", "APOE", "MLH1"]; dis_t = ["BC", "AD", "T2D"]
E_GD = [("BRCA1", "BC"), ("TP53", "BC"), ("TP53", "AD"), ("APOE", "AD"), ("APOE", "T2D"), ("MLH1", "BC"), ("MLH1", "T2D")]
E_DD = [("BC", "AD")]

def toy(e_gd, e_dd, genes, dis):
    A_GD = np.zeros((len(genes), len(dis))); A_DD = np.zeros((len(dis), len(dis)))
    for g, d in e_gd: A_GD[genes.index(g), dis.index(d)] = 1
    for a, b in e_dd: A_DD[dis.index(a), dis.index(b)] = 1
    return {("Gene", "Disease"): sp.csr_matrix(A_GD), ("Disease", "Disease"): sp.csr_matrix(A_DD)}

A_toy = toy(E_GD, E_DD, genes_t, dis_t)
W_toy = weighted(A_toy, 0.5)
D = dwpc_matrix(METAPATHS["Gene>Disease>Disease"], W_toy).toarray()
print("DWPC, rows = genes, columns = BC, AD, T2D:\n", D.round(3))
print("DWPC(BRCA1, AD) =", D[0, 1].round(3))

**Q3.** Without MLH1, BRCA1's DWPC to AD would be $1^{-1/2}\cdot2^{-1/2}=0.707$. After adding MLH1 it fell to 0.577, although BRCA1's own edges did not change. Why?

**Expected output.** DWPC matrix (columns BC, AD, T2D): BRCA1 $(0,0.577,0)$, TP53 $(0,0.408,0)$, APOE $(0,0,0)$, MLH1 $(0,0.408,0)$; and `DWPC(BRCA1, AD) = 0.577`.

**Solution Q3.** Adding MLH1 → BC raised BC's in-degree from 2 to 3. Every path through BC now carries a factor $3^{-1/2}=0.577$ instead of $2^{-1/2}=0.707$. DWPC weights depend on the *global* degree of the nodes on the path, so adding a node changes the weights of paths between other nodes: BC has become more of a hub, so it is discounted more.

---
# Exercise 3: An ROC curve by hand  *(8 min)*

A model scored six gene–disease pairs. Three are truly positive:

| score | 0.9 | 0.8 | 0.7 | 0.6 | 0.4 | 0.3 |
|---|---|---|---|---|---|---|
| true label | 1 | 1 | 0 | 1 | 0 | 0 |

For a threshold $\tau$ a pair is predicted positive if its score is $\ge\tau$. Then
$$\mathrm{TPR}=\frac{TP}{\text{all positives}},\qquad \mathrm{FPR}=\frac{FP}{\text{all negatives}}.$$

**By hand.**

1. For $\tau=0.9,\ 0.8,\ 0.7,\ 0.6,\ 0.4,\ 0.3$ fill in a table of TP, FP, TPR, FPR.
2. Sketch the ROC curve: FPR on the horizontal axis, TPR on the vertical axis, starting at (0, 0).
3. The AUROC is the probability that a randomly chosen positive is scored above a randomly chosen negative. There are $3\times3=9$ positive–negative pairs; in how many is the positive scored higher? Give the AUROC as that fraction.

**Solution.**

| $\tau$ | TP | FP | TPR | FPR |
|---|---|---|---|---|
| 0.9 | 1 | 0 | 0.33 | 0 |
| 0.8 | 2 | 0 | 0.67 | 0 |
| 0.7 | 2 | 1 | 0.67 | 0.33 |
| 0.6 | 3 | 1 | 1.00 | 0.33 |
| 0.4 | 3 | 2 | 1.00 | 0.67 |
| 0.3 | 3 | 3 | 1.00 | 1.00 |

The curve goes up the left edge to (0, 0.67), right to (0.33, 0.67), up to (0.33, 1), then right along the top to (1, 1).

AUROC: the positives scored 0.9 and 0.8 beat all three negatives (6 pairs); the positive scored 0.6 beats two (0.4 and 0.3) but loses to the 0.7 negative (2 pairs). That is $8/9=\mathbf{0.889}$. (Area under the curve gives the same: $\tfrac13\cdot\tfrac23+\tfrac13+\tfrac13=0.889$.)

**Check in code.** Run the cell. Do the AUROC and the curve points match yours?

In [ ]:
from sklearn.metrics import roc_curve
scores = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.3]); labels = np.array([1, 1, 0, 1, 0, 0])
print("AUROC =", round(roc_auc_score(labels, scores), 3))
fpr_, tpr_, thr_ = roc_curve(labels, scores, drop_intermediate=False)
print(pd.DataFrame({"threshold": thr_, "FPR": fpr_.round(2), "TPR": tpr_.round(2)}))

**Q4.** If the negative scored 0.7 were instead scored 0.5, what would the AUROC be?

**Q5 (think, no calculation).** On HALD there are 132 positive and 20,000 negative pairs. A model can have an AUROC of 0.99 and still be wrong about most of the pairs it flags when we allow 1% false positives. Why?

**Expected output.** `AUROC = 0.889`, and the curve points (FPR, TPR) from (0, 0) through (0, 0.33), (0, 0.67), (0.33, 0.67), (0.33, 1), (0.67, 1) to (1, 1); `thresholds` are listed with an extra first row.

**Solution Q4.** Every positive would now outrank every negative: $9/9=\mathbf{1.0}$.

**Solution Q5.** A 1% false positive rate means 1% of the 20,000 negatives, i.e. 200 false alarms, which is more than the 132 positives that exist. So even if most positives are found, most flagged pairs are false. The AUROC averages over all thresholds and hides this class imbalance, which is why average precision (or precision in the top predictions) is also reported. (On the real HALD model, 114 true positives and 200 false positives at 1% FPR gives a precision of about 36%.)

---
# Exercise 4: Fairer negatives  *(7 min)*

In the main notebook the negatives are random gene–disease pairs. Positive genes and diseases are mostly high-degree "hubs", so a model can do well just by recognising hubs. Here we build **degree-matched negatives**: for each positive pair $(g,d)$ we sample 20 negative pairs whose gene has a similar total degree to $g$ and whose disease has a similar total degree to $d$ (same degree decile).

**Step 1.** Run this cell as it is (about 30 seconds). It builds the matched negatives and their features.

In [ ]:
def total_degree(cls):
    d = np.zeros(len(Nodes[cls]))
    for (a, b), M in A0.items():
        if a == cls: d += np.asarray(M.sum(1)).ravel()          # edges leaving a node of this class
        if b == cls: d += np.asarray(M.sum(0)).ravel()          # edges arriving at a node of this class
    return d

gdeg, ddeg = total_degree("Gene"), total_degree("Disease")
decile = lambda deg: pd.qcut(pd.Series(deg).rank(method="first"), 10, labels=False).to_numpy()
gbin, dbin = decile(gdeg), decile(ddeg)                         # degree decile (0..9) of every gene / disease
genes_in = {b: np.where(gbin == b)[0] for b in range(10)}
dis_in   = {b: np.where(dbin == b)[0] for b in range(10)}

rng7, neg_m = np.random.default_rng(7), set()
for g, d in pos_idx:
    chosen, tries = set(), 0
    while len(chosen) < 20 and tries < 5000:
        tries += 1
        cand = (int(rng7.choice(genes_in[gbin[g]])), int(rng7.choice(dis_in[dbin[d]])))
        if cand not in posset and cand not in chosen: chosen.add(cand)
    neg_m |= chosen

PAIRS_M = np.vstack([pos_idx, np.array(sorted(neg_m))])
Y_M = np.r_[np.ones(len(pos_idx)), np.zeros(len(PAIRS_M) - len(pos_idx))].astype(int)
X_M  = features(A0, W_BEST, NAMES, PAIRS_M)                    # DWPC features for the matched pairs
XD_M = degree_features(A0, PAIRS_M)[0][:, kd]                  # degree covariates for the matched pairs
print("median total degree of positive genes:", np.median(gdeg[pos_idx[:, 0]]), " of all genes:", np.median(gdeg))
print("matched dataset:", PAIRS_M.shape[0], "pairs,", Y_M.sum(), "positive")

**Step 2.** `cv_oof` uses the global labels `Y`, so here is a version that takes the labels as an argument. Complete the one missing line, then run. It compares three models with random negatives (as in the main notebook) and with matched negatives.

In [ ]:
def cv_eval(Xm, y, mk):
    oof = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xm, y):
        oof[te] = mk().fit(Xm[tr], y[tr]).predict_proba(Xm[te])[:, 1]
    return roc_auc_score(y, oof)                               # AUROC only

mk7 = lambda: make_ridge(C=1e-4)
for label, (Xr, Xm) in {"degree only": (XD, XD_M), "DWPC": (X, X_M), "DWPC + degree": (np.hstack([X, XD]), np.hstack([X_M, XD_M]))}.items():
    auc_random  = cv_eval(Xr, Y, mk7)
    auc_matched = cv_eval(Xm, Y_M, mk7)
    print(f"{label:14s}  AUROC random negatives = {auc_random:.3f}   matched negatives = {auc_matched:.3f}")

**Q6.** What happens to the AUROC of the **degree-only** model when the negatives are degree-matched? Why does it not fall all the way to 0.5?

**Q7.** In one or two sentences: why is the matched-negative result a better test of whether the metapath features are useful?

**Expected output.** Median total degree of positive genes: 35 (all genes: 1). Matched dataset: 2,760 pairs, 132 positive.

| features | AUROC random negatives | AUROC matched negatives |
|---|---|---|
| degree only | 0.983 | 0.871 |
| DWPC | 0.977 | 0.855 |
| DWPC + degree | 0.987 | 0.882 |

**Solution Q6.** The degree-only AUROC falls from 0.983 to **0.871**, a large drop, but not to 0.5. We matched on one summary (total degree decile), so the 37 per-metaedge degree features still carry information (for example which *kinds* of edges a node has, and degree still varies within a decile).

**Solution Q7.** With random negatives, a model can score highly just by detecting hub genes and hub diseases, without using any specific link between the gene and the disease. Matched negatives remove most of that shortcut, so the AUROC that remains is closer to the evidence that the *particular* pair is connected. Here the metapath features keep a clear signal (0.855 on their own).